In [2]:
import fitz
import os
import re
import json
import glob
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

KeyboardInterrupt: 

In [2]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    
    for page in doc:
        text += page.get_text()
    
    return text

def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    
    for file in os.listdir(input_folder):
        if file.endswith(".PDF") or file.endswith(".pdf"):
            path = os.path.join(input_folder, file)
            text = extract_text_from_pdf(path)
            
            with open(os.path.join(output_folder, file.replace(".PDF", ".txt")), "w", encoding="utf-8") as f:
                f.write(text)

def clean_text(text):
    text = re.sub(r'\n+', '\n', text)  # remove excessive newlines
    text = re.sub(r'\s+', ' ', text)   # normalize spaces
    return text.strip()

In [33]:
for i in tqdm(range(2020, 2026)):
    input_folder = f"data/supreme_court_judgments/{i}"
    output_folder = f"data/extracted_data/{i}"
    process_folder(input_folder, output_folder)

100%|██████████| 6/6 [01:19<00:00, 13.21s/it]


In [1]:
with open("data/extracted_data/2020/Abhilasha_vs_Parkash_on_15_September_2020_1.txt", "r", encoding="utf-8") as f:
    sample_text = f.read()

In [9]:
text = preprocess_raw_text(sample_text)
with open('tmp.txt', 'w', encoding='utf-8') as f:
    f.write(text)

In [3]:
def fix_spaced_text(text):
    # matches sequences like "h i s" or "f a t h e r"
    pattern = r'(?:\b[a-zA-Z]\s){2,}[a-zA-Z]\b'

    def replacer(match):
        return match.group(0).replace(' ', '')

    return re.sub(pattern, replacer, text)

def preprocess_raw_text(text):
    
    text = re.sub(r'Equivalent citations:.*?\d{4}', '', text, flags=re.IGNORECASE) # Remove citation block
    text = re.sub(r'IN THE SUPREME COURT OF INDIA.*?JURISDICTION', '', text, flags=re.IGNORECASE) # Remove court header
    text = re.sub(r'REPORTABL[E]?', '', text, flags=re.IGNORECASE) # Remove REPORTABLE / REPORTABL noise
    text = fix_spaced_text(text) # Fix spaced-out words like J U D G M E N T
    text = re.sub(r'\s+', ' ', text) # Normalize whitespace
    text = re.sub(r'\.\.\.APPELLANT\(S\).*?RESPONDENT\(S\)', '', text, flags=re.IGNORECASE) # Remove APPELLANT / RESPONDENT block (optional but helpful)
    text = re.sub(r'Indian Kanoon - http[s]?://\S+', '', text) # Remove indiakanoon.com hyperlink
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\b[A-Z][a-zA-Z\s]+ vs [A-Z][a-zA-Z\s]+ on \d{1,2} \w+, \d{4}', '', text)

    preamble = text[:1000].lower()
    idx_judgment = preamble.find("judgment")
    idx_order = preamble.find("order")
    indices = [i for i in [idx_judgment, idx_order] if i != -1]

    if indices:
        split_idx = min(indices)
        text = text[split_idx + len("judgment"):] if split_idx == idx_judgment else text[split_idx + len("order"):]

    return text.strip()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,       # characters, not tokens (good enough)
    chunk_overlap=300
)

def create_chunks(text):
    return splitter.split_text(text)

def parse_filename(filename):
    # Abhilasha_vs_Parkash_on_15_September_2020_1
    basename = filename.split("/")[-1]
    parts = basename.replace(".txt", "").split("_on_")
    
    case_name = parts[0].replace("_vs_", " vs ").replace("_", " ")
    date_part = parts[1].rsplit("_", 1)[0].replace("_", " ")
    
    return case_name, date_part

def build_chunks(filename):

    case_name, date = parse_filename(filename)
    with open(filename, "r", encoding="utf-8") as f:
        raw_text = f.read()
    text = preprocess_raw_text(raw_text)
    raw_chunks = create_chunks(text)

    structured_chunks = []

    for chunk in raw_chunks:
        enriched_text = f"""
Case: {case_name}
Date: {date}

{chunk}
"""
        structured_chunks.append({
            "id": f"{case_name}_{date}_{len(structured_chunks)}",
            "text": enriched_text,
            "case_name": case_name,
            "date": date
        })

    return structured_chunks

In [5]:
chunks = build_chunks("./data/extracted_data/2020/Abhilasha_vs_Parkash_on_15_September_2020_1.txt")

print(chunks[0]["text"][:500])
print(len(chunks))


Case: Abhilasha vs Parkash
Date: 15 September 2020

ASHOK BHUSHAN,J. Leave granted. 2. This appeal has been filed by the appellant, daughter of respondent Nos. 1 and 2, challenging the order of the High Court of Punjab and Haryana at Chandigarh dated 16.08.2018 by which order the High Court dismissed the application under Section 482 Cr.P.C. filed by the appellant praying for setting aside the order of the Judicial Magistrate First Class, Rewari dated 16.02.2011 as well as the order dated 17.02
26


In [5]:
def save_chunks_json(all_chunks, file_path):
    with open(file_path, "w", encoding="utf-8") as f:
        for chunk in all_chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

def load_chunks_json(file_path):
    chunks = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))
    return chunks

In [7]:
files = glob.glob("data/extracted_data/2025/*.txt")
files = [f.replace("\\", "/") for f in files]
all_chunks = []

for file in files:
    chunks = build_chunks(file)
    all_chunks.extend(chunks)
len(all_chunks)

7803

In [8]:
save_chunks_json(all_chunks, "./data/chunked_data/chunks_v2_400docs.json")

In [ ]:
chunks = load_chunks_json("./data/chunked_data/chunks_v2_20docs.json")

377

In [98]:
with open(file, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
tmp = preprocess_raw_text(raw_text)

In [106]:
tmp.find('ingredients')

5844

In [1]:
import glob
len(glob.glob("./data/supreme_court_judgments/*/*.PDF"))

26688